# 🔪 The Complete Guide to Chunking Strategies for RAG

## What You'll Learn

In this notebook, you'll implement and compare **six different chunking strategies** 
side by side, using the same document and the same evaluation harness.

### The Six Strategies:
1. 🟢 **Fixed-Size Chunking** — Split every N characters
2. 🟡 **Recursive Character Chunking** — Split at natural boundaries
3. 🔵 **Sentence-Window Chunking** — Embed small, retrieve big
4. 🟣 **Semantic Chunking** — Split where meaning shifts
5. 🟠 **Document-Aware Chunking** — Follow the document's structure
6. 🔴 **Code-Aware Chunking** — Respect code syntax boundaries

### How to Use This Notebook:
- Run cells **top to bottom** (setup cells are required for everything below)
- Each strategy is **self-contained** — you can skip to any one
- The **Grand Comparison** at the end runs all strategies head-to-head
- **Experiment!** Change parameters, swap documents, and see what happens

### Document Used:
We use the **WHO Global Action Plan on Antimicrobial Resistance** throughout.
Every strategy chunks the same document so you can compare results directly.

> 📥 Download the PDF: https://iris.who.int/server/api/core/bitstreams/6989e26c-c181-4ec8-bb99-104415a2e142/content  
> Save it as `who_amr_action_plan.pdf` in the same folder as this notebook.


---
## 📦 Step 0: Install Dependencies

Run this cell once. Restart the kernel after installation if needed.


In [ ]:
import os
# Prevent transformers/sentence-transformers from loading TF/Keras (PyTorch only)
os.environ["USE_TF"] = "0"


In [ ]:
# !pip install langchain langchain-community pypdf sentence-transformers chromadb numpy


---
## 🔧 Step 1: Setup — Load Document & Build Evaluation Harness

These two cells set up everything we need. Every strategy below depends on them.


In [ ]:
# ============================================================
# 1A: LOAD THE DOCUMENT
# ============================================================

import os
import re
import shutil
import warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import PyPDFLoader
from langchain.schema import Document

# ------------------------------------------------------------------
# 📥 CONFIGURE YOUR PDF PATH HERE
# ------------------------------------------------------------------
PDF_PATH = "data/who_document.pdf"

if not os.path.exists(PDF_PATH):
    print("⚠️  PDF not found! Please download it:")
    print("    https://iris.who.int/server/api/core/bitstreams/6989e26c-c181-4ec8-bb99-104415a2e142/content")
    print(f"    Save as: {PDF_PATH}")
else:
    loader = PyPDFLoader(PDF_PATH)
    pages = loader.load()

    print(f"✅ Loaded {len(pages)} pages")
    print(f"\n📄 First page preview ({len(pages[0].page_content)} chars):")
    print("-" * 60)
    print(pages[0].page_content[:500])
    print("-" * 60)
    print(f"\n📎 Metadata: {pages[0].metadata}")


In [ ]:
# ============================================================
# 1B: COMMON EVALUATION HARNESS
# ============================================================
# This function takes ANY list of chunks, builds a temporary IN-MEMORY
# vector store, runs test queries, and prints results.
# Using EphemeralClient (in-memory) avoids disk I/O errors entirely.
# ============================================================

import chromadb
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Same embedding model for ALL strategies (apples-to-apples comparison)
print("Loading embedding model (first time may take a minute)...")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": DEVICE}
)
print(f"✅ Embedding model loaded on {DEVICE}!\n")

# Test queries — real questions someone might ask about this document
TEST_QUERIES = [
    "What are the strategic objectives of the global action plan?",
    "How does antimicrobial resistance spread between animals and humans?",
    "What does the WHO recommend for surveillance of resistant infections?",
    "What role do member states play in combating AMR?",
    "How should antibiotics be regulated in food-producing animals?",
]


def evaluate_chunking_strategy(chunks, strategy_name, queries=TEST_QUERIES, top_k=3):
    """
    Takes a list of chunked Documents, builds a temporary in-memory vector
    store, runs test queries, and prints results for comparison.

    Args:
        chunks: list of LangChain Document objects
        strategy_name: string label for this strategy
        queries: list of test query strings
        top_k: number of results to retrieve per query

    Returns:
        dict with chunk stats and retrieval results for further analysis
    """
    print(f"\n{'='*70}")
    print(f"📊 STRATEGY: {strategy_name}")
    print(f"{'='*70}")

    # --- Chunk Statistics ---
    chunk_sizes = [len(c.page_content) for c in chunks]
    stats = {
        "name": strategy_name,
        "count": len(chunks),
        "avg_size": sum(chunk_sizes) // len(chunk_sizes),
        "min_size": min(chunk_sizes),
        "max_size": max(chunk_sizes),
    }

    print(f"\n  📦 Chunk count:    {stats['count']}")
    print(f"  📏 Avg chunk size: {stats['avg_size']} chars")
    print(f"  📐 Min chunk size: {stats['min_size']} chars")
    print(f"  📐 Max chunk size: {stats['max_size']} chars")

    # --- Show First 3 Chunks ---
    print(f"\n  --- First 3 Chunks ---")
    for i, chunk in enumerate(chunks[:3]):
        preview = chunk.page_content[:150].replace('\n', ' ')
        print(f"\n  📄 Chunk {i} ({len(chunk.page_content)} chars):")
        print(f"     \"{preview}...\"")
        if chunk.metadata:
            clean_meta = {
                k: (str(v)[:80] + "..." if len(str(v)) > 80 else v)
                for k, v in chunk.metadata.items()
                if k != "window"  # skip window text (too long)
            }
            print(f"     Metadata: {clean_meta}")

    # --- Build In-Memory Vector Store (no disk writes, no SQLite errors) ---
    chroma_client = chromadb.EphemeralClient()
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        client=chroma_client,
    )

    # --- Run Test Queries ---
    print(f"\n  --- Retrieval Results (top-{top_k} per query) ---")
    all_results = {}

    for query in queries:
        results = vectorstore.similarity_search_with_score(query, k=top_k)
        all_results[query] = results

        print(f"\n  🔍 Q: \"{query}\"")
        for j, (doc, score) in enumerate(results):
            preview = doc.page_content[:120].replace('\n', ' ')
            print(f"     [{j+1}] (distance: {score:.4f}) \"{preview}...\"")

    # EphemeralClient is garbage-collected automatically — no cleanup needed

    stats["results"] = all_results
    return stats


print("✅ Evaluation harness ready!")
print(f"📝 Test queries loaded: {len(TEST_QUERIES)}")
print("\n💡 Usage: evaluate_chunking_strategy(your_chunks, 'Strategy Name')")


---
---
## 🟢 Strategy 1: Fixed-Size Chunking

**The idea:** Split every N characters. That's it.

This is the "I just need chunks and I need them now" approach. You pick a number — 
say 500 characters — and you chop the document every 500 characters. Optionally, 
you add **overlap** so the end of one chunk repeats at the beginning of the next.
Pros: Dead simple. Fast. Predictable chunk sizes. Cons: Breaks mid-sentence, mid-word, mid-thought. Blind to meaning. Use: Homogeneous prose where boundaries matter less (essays, transcripts).


In [ ]:
# ============================================================
# STRATEGY 1: Fixed-Size Chunking
# ============================================================

from langchain.text_splitter import CharacterTextSplitter

# ------------------------------------------------------------------
# 🎛️ EXPERIMENT: Try changing these parameters!
#    - chunk_size: 200, 500, 1000
#    - chunk_overlap: 0, 50, 100
# ------------------------------------------------------------------
fixed_splitter = CharacterTextSplitter(
    separator="",           # No separator — pure character count
    chunk_size=500,         # 500 characters per chunk
    chunk_overlap=50,       # 50-character overlap between chunks
    length_function=len,
)

fixed_chunks = fixed_splitter.split_documents(pages)
fixed_stats = evaluate_chunking_strategy(fixed_chunks, "Fixed-Size (500 chars)")


### 🔍 What to Notice:
- Look at the chunk previews. Do any of them start or end mid-sentence?
- Check the retrieval results. Are the top results actually answering the question?
- Compare the distance scores — lower distance = better match in Chroma.

### 🧪 Try This:
Change `chunk_size` to 200 and re-run. Then try 1000. How does chunk size 
affect the number of chunks and retrieval quality?


---
## 🟡 Strategy 2: Recursive Character Chunking

**The idea:** Try to split at natural boundaries, falling back to smaller 
boundaries only when needed.

The hierarchy: paragraphs → newlines → sentences → words → characters.

This is the **default strategy in most RAG frameworks** and your recommended 
starting point.


In [ ]:
# ============================================================
# STRATEGY 2: Recursive Character Chunking
# ============================================================

from langchain.text_splitter import RecursiveCharacterTextSplitter

# ------------------------------------------------------------------
# 🎛️ EXPERIMENT: Try changing these parameters!
#    - chunk_size: 200, 500, 1000
#    - chunk_overlap: 0, 50, 100
#    - separators: reorder or remove items to see what changes
# ------------------------------------------------------------------
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,           # Target chunk size in characters
    chunk_overlap=50,         # Overlap to preserve context at boundaries
    separators=[              # Tried in order — falls back to the next if chunk still too large
        "\n\n",            # 1st choice: paragraph break
        "\n",               # 2nd choice: line break
        ". ",                # 3rd choice: sentence boundary
        " ",                 # 4th choice: word boundary
        "",                  # Last resort: character split
    ],
    length_function=len,
)

recursive_chunks = recursive_splitter.split_documents(pages)

# --- Show how it respects boundaries vs. fixed-size ---
print(f"\n{'='*70}")
print(f"📊 STRATEGY: Recursive Character Chunking")
print(f"{'='*70}")
print(f"\n  Fixed-size chunks: {len(fixed_chunks)}  |  Recursive chunks: {len(recursive_chunks)}")
print(f"\n  --- First 3 Recursive Chunks ---")
for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"\n  Chunk {i} ({len(chunk.page_content)} chars):")
    print(f"  \"{chunk.page_content[:200].replace(chr(10), ' ')}...\"")

recursive_stats = evaluate_chunking_strategy(recursive_chunks, "Recursive Character (500 chars)")


---
## 🔵 Strategy 3: Sentence-Window Chunking

**The idea:** Embed small (precise), retrieve big (contextual).

Each chunk is a **single sentence** — this is what gets turned into a vector and matched against your query. But when a sentence is retrieved, you pass a wider **window of surrounding sentences** to the LLM, so it has enough context to answer.

```
Document:  [S1] [S2] [S3] [S4] [S5] [S6] [S7]
                       ↑
              Query matches S4

Embedded:  "S4"                          ← precise match
Retrieved: "S3  S4  S5"  (window=1)     ← contextual answer
```

**When to use it:**
- Documents with dense, factual prose (reports, papers, guidelines)
- When you need pinpoint retrieval *and* enough context for a good answer
- When fixed/recursive chunks feel either too small (lose context) or too large (dilute the match)

**Trade-offs:**

| Pros | Cons |
|---|---|
| Highly precise embedding targets | Many more vectors to store |
| LLM gets contextual window | Tuning `window_size` adds complexity |
| Works well on factual Q&A | Short sentences can be noisy |


In [ ]:
# ============================================================
# STRATEGY 3: Sentence-Window Chunking
# ============================================================

def sentence_window_chunker(documents, window_size=3):
    """
    Split into individual sentences for embedding,
    but attach a surrounding window of sentences as retrievable context.
    
    Each chunk's page_content = single sentence (what gets EMBEDDED)
    Each chunk's metadata['window'] = surrounding sentences (what gets RETRIEVED)
    
    Args:
        documents: list of LangChain Document objects
        window_size: number of sentences before/after to include in window
    
    Returns:
        list of Document objects (one per sentence)
    """
    all_chunks = []
    
    for doc in documents:
        text = doc.page_content
        
        # Split into sentences (simple regex — works for most prose)
        sentences = re.split(r'(?<=[.!?])\s+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        for i, sentence in enumerate(sentences):
            # Skip very short fragments (headers, page numbers, etc.)
            if len(sentence) < 20:
                continue
            
            # Build the surrounding window
            start = max(0, i - window_size)
            end = min(len(sentences), i + window_size + 1)
            window_text = " ".join(sentences[start:end])
            
            chunk = Document(
                page_content=sentence,  # ← This gets EMBEDDED (precise)
                metadata={
                    **doc.metadata,
                    "window": window_text,       # ← This gets RETRIEVED (contextual)
                    "sentence_index": i,
                    "total_sentences": len(sentences),
                    "window_start": start,
                    "window_end": end,
                }
            )
            all_chunks.append(chunk)
    
    return all_chunks


# ------------------------------------------------------------------
# 🎛️ EXPERIMENT: Try changing window_size!
#    - window_size=1: minimal context (3 sentences total)
#    - window_size=3: moderate context (7 sentences total)
#    - window_size=5: wide context (11 sentences total)
# ------------------------------------------------------------------
sentence_window_chunks = sentence_window_chunker(pages, window_size=3)

# --- Show the embed vs. retrieve distinction ---
print(f"\n{'='*70}")
print(f"📊 STRATEGY: Sentence-Window (window=3)")
print(f"{'='*70}")
print(f"\n  Total sentence chunks: {len(sentence_window_chunks)}")

if len(sentence_window_chunks) > 10:
    example = sentence_window_chunks[10]
    print(f"\n  💡 KEY CONCEPT: Embed vs. Retrieve")
    print(f"  {'─'*50}")
    print(f"  🎯 EMBEDDED (single sentence):")
    print(f"     \"{example.page_content[:150]}\"")
    print(f"\n  📦 RETRIEVED (window of sentences):")
    print(f"     \"{example.metadata['window'][:300]}...\"")
    print(f"\n  The embedding is precise. The retrieval is contextual.")

# Evaluate (note: we embed the sentences, not the windows)
sw_stats = evaluate_chunking_strategy(
    sentence_window_chunks, 
    "Sentence-Window (window=3)"
)


### 🔍 What to Notice:
- The chunk count is MUCH higher (one per sentence vs. one per paragraph).
- Each "chunk" is a single sentence — look at how small they are.
- The retrieval distance scores might be lower (better) because sentence-level 
  matching is more precise.
- In production, you'd send the `metadata['window']` to the LLM, not just the sentence.

### 🧪 Try This:
1. Change `window_size` to 1 and then 5. How does the window content change?
2. Look at the example chunk — does the window provide enough context for an LLM?


In [ ]:
# ============================================================
# STRATEGY 4: Semantic Chunking
# ============================================================

import numpy as np
from sentence_transformers import SentenceTransformer

# We use the same model for consistency, but you could use any sentence-transformer
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")


def semantic_chunker(documents, similarity_threshold=0.75, min_chunk_size=100):
    """
    Split documents based on semantic similarity between adjacent sentences.
    When similarity drops below the threshold, a new chunk begins.
    
    Args:
        documents: list of LangChain Document objects
        similarity_threshold: similarity below this triggers a split.
                              Lower = fewer splits (bigger chunks).
                              Higher = more splits (smaller chunks).
        min_chunk_size: minimum characters per chunk (prevents tiny fragments)
    
    Returns:
        list of Document objects (one per semantic segment)
    """
    all_chunks = []
    
    for doc_idx, doc in enumerate(documents):
        text = doc.page_content
        
        # Step 1: Split into sentences
        sentences = re.split(r'(?<=[.!?])\s+', text)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
        
        if len(sentences) < 2:
            # Too few sentences to analyze — keep as one chunk
            if text.strip():
                all_chunks.append(doc)
            continue
        
        # Step 2: Embed every sentence
        embeddings = semantic_model.encode(sentences)
        
        # Step 3: Compare adjacent sentences using cosine similarity
        similarities = []
        for i in range(len(embeddings) - 1):
            sim = np.dot(embeddings[i], embeddings[i + 1]) / (
                np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i + 1])
            )
            similarities.append(float(sim))
        
        # Step 4: Find split points where similarity drops below threshold
        split_points = []
        for i, sim in enumerate(similarities):
            if sim < similarity_threshold:
                split_points.append(i + 1)  # split AFTER sentence i
        
        # Step 5: Group sentences into chunks based on split points
        current_start = 0
        for split_idx in split_points:
            chunk_text = " ".join(sentences[current_start:split_idx])
            
            # Enforce minimum chunk size — merge tiny chunks forward
            if len(chunk_text) < min_chunk_size:
                continue  # skip this split, let it merge with the next group
            
            all_chunks.append(Document(
                page_content=chunk_text,
                metadata={
                    **doc.metadata,
                    "chunk_type": "semantic",
                    "sentence_range": f"{current_start}-{split_idx}",
                    "num_sentences": split_idx - current_start,
                }
            ))
            current_start = split_idx
        
        # Don't forget the last chunk
        if current_start < len(sentences):
            remaining = " ".join(sentences[current_start:])
            if remaining.strip() and len(remaining) >= min_chunk_size:
                all_chunks.append(Document(
                    page_content=remaining,
                    metadata={
                        **doc.metadata,
                        "chunk_type": "semantic",
                        "sentence_range": f"{current_start}-{len(sentences)}",
                        "num_sentences": len(sentences) - current_start,
                    }
                ))
    
    return all_chunks


# ------------------------------------------------------------------
# 🎛️ EXPERIMENT: Try changing the similarity threshold!
#    - 0.5: very few splits (large, multi-topic chunks)
#    - 0.75: moderate splits (balanced — good default)
#    - 0.85: many splits (small, focused chunks)
#    - 0.9: aggressive splits (almost sentence-level)
# ------------------------------------------------------------------
print("⏳ Running semantic chunking (this takes a minute — embedding every sentence)...")
semantic_chunks = semantic_chunker(pages, similarity_threshold=0.75)
print(f"✅ Done! Created {len(semantic_chunks)} semantic chunks.\n")

semantic_stats = evaluate_chunking_strategy(semantic_chunks, "Semantic (threshold=0.75)")


In [ ]:
# ============================================================
# 🔬 BONUS: Visualize the Similarity Landscape
# ============================================================
# This shows you WHY the semantic chunker split where it did.
# Look for the dips — those are topic boundaries.
# ============================================================

def visualize_similarity_landscape(doc, threshold=0.75, max_sentences=30):
    """
    Show the sentence-by-sentence similarity scores for a single page.
    Dips below the threshold = topic boundaries = split points.
    """
    text = doc.page_content
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    
    if len(sentences) < 3:
        print("  Too few sentences on this page to visualize.")
        return
    
    # Limit for readability
    sentences = sentences[:max_sentences]
    
    embeddings = semantic_model.encode(sentences)
    
    print(f"\n  📈 Similarity Landscape (Page {doc.metadata.get('page', '?')})")
    print(f"  {'─'*60}")
    print(f"  Threshold: {threshold} (splits happen below this line)")
    print(f"  Sentences analyzed: {len(sentences)}\n")
    
    for i in range(len(embeddings) - 1):
        sim = np.dot(embeddings[i], embeddings[i + 1]) / (
            np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i + 1])
        )
        
        # Visual bar
        bar_length = int(sim * 40)
        bar = "█" * bar_length + "░" * (40 - bar_length)
        
        # Mark splits
        marker = " ✂️  SPLIT" if sim < threshold else ""
        
        # Sentence preview
        preview = sentences[i][:45].replace('\n', ' ')
        
        print(f"  S{i:2d} │{bar}│ {sim:.3f}{marker}")
        print(f"       \"{preview}...\"")
    
    print(f"\n  💡 Each ✂️ marks where the semantic chunker would split.")
    print(f"     The dips show where the document shifts topic.")


# Visualize a few interesting pages
print("\n" + "="*70)
print("🔬 SIMILARITY LANDSCAPE VISUALIZATION")
print("="*70)

# Try pages that likely have topic shifts
for page_num in [3, 5, 10]:
    if page_num < len(pages):
        visualize_similarity_landscape(pages[page_num], threshold=0.75)


### 🔍 What to Notice:
- The similarity landscape shows you the "terrain" of your document. High similarity = 
  same topic. Sharp dips = topic changes.
- Chunk sizes are **unpredictable** — some chunks might be 2 sentences, others 15.
- Compare the chunk count with recursive character chunking. Which produces more chunks?
- Check if the retrieval results are more relevant than simpler strategies.

### 🧪 Try This:
1. Change `similarity_threshold` to 0.5 (fewer splits) and 0.85 (more splits). 
   How does it affect chunk count and retrieval?
2. Visualize different pages — which pages have the most topic shifts?
3. Try a different embedding model (e.g., `all-mpnet-base-v2`) — do the splits change?


---
## 🟠 Strategy 5: Document-Aware Chunking

**The idea:** Use the document's own structure — headers, sections, bullet points — 
as natural split points.

Why guess where to split when the **document is already telling you**?

We show two approaches:
- **5A: Markdown-based** — for clean structured text
- **5B: PDF heuristic** — for real-world PDFs where structure must be detected


In [ ]:
# ============================================================
# STRATEGY 5A: Document-Aware Chunking (Markdown)
# ============================================================
# If your documents are Markdown (or you can convert them),
# this approach gives you the cleanest, most metadata-rich chunks.
# ============================================================

from langchain.text_splitter import MarkdownHeaderTextSplitter

# --- Sample Markdown (simulating a well-structured document) ---
SAMPLE_MARKDOWN = """
# Global Action Plan on Antimicrobial Resistance

## Executive Summary

Antimicrobial resistance (AMR) threatens the effective prevention and treatment 
of an ever-increasing range of infections. It is an increasingly serious threat 
to global public health that requires action across all government sectors and 
society. The cost of AMR to national economies and their health systems is 
significant as it affects productivity of patients or their caretakers.

## Strategic Objectives

### Objective 1: Awareness and Understanding

Improve awareness and understanding of antimicrobial resistance through effective 
communication, education and training. AMR and the measures to combat it should 
be widely understood. Public awareness campaigns should target different audiences 
including prescribers, farmers, and the general public.

### Objective 2: Surveillance and Research

Strengthen the knowledge and evidence base through surveillance and research. 
National and international surveillance systems must be strengthened to monitor 
trends in resistance. Data sharing between countries is essential for tracking 
the global spread of resistant organisms.

### Objective 3: Infection Prevention

Reduce the incidence of infection through effective sanitation, hygiene and 
infection prevention measures. Clean water, sanitation, and hygiene in healthcare 
facilities and community settings reduce the spread of infections, decreasing 
the need for antibiotics.

## Implementation Framework

### Role of Member States

Each member state should develop a national action plan on AMR aligned with 
this global action plan. Plans should be multisectoral, covering human health, 
animal health, agriculture, and the environment.

### Funding and Resources

Sustainable financing mechanisms are needed to support AMR activities. Countries 
should allocate domestic resources and seek international support where needed.

## Monitoring and Evaluation

Progress should be monitored through agreed indicators at national and global 
levels. Regular reporting to WHO will enable tracking of implementation.
"""

# Define which headers to split on
headers_to_split_on = [
    ("#", "h1"),       # Top-level header
    ("##", "h2"),      # Section header
    ("###", "h3"),     # Sub-section header
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,  # Keep headers in the chunk text for embedding
)

md_header_chunks = markdown_splitter.split_text(SAMPLE_MARKDOWN)

# Convert to Document objects for our evaluation harness
md_chunks = [
    Document(
        page_content=chunk.page_content,
        metadata=chunk.metadata
    )
    for chunk in md_header_chunks
]

print(f"\n{'='*70}")
print(f"📊 STRATEGY: Document-Aware (Markdown Headers)")
print(f"{'='*70}")
print(f"\n  Chunks created: {len(md_chunks)}")

for i, chunk in enumerate(md_chunks):
    print(f"\n  📄 Chunk {i} ({len(chunk.page_content)} chars):")
    print(f"     Headers: {chunk.metadata}")
    preview = chunk.page_content[:200].replace('\n', ' ')
    print(f"     Content: \"{preview}...\"")

print(f"\n  💡 Notice how each chunk carries its section hierarchy in metadata!")
print(f"     This enables powerful filtering: 'only search in Implementation Framework'")


In [ ]:
# ============================================================
# STRATEGY 5B: Document-Aware Chunking (PDF Heuristic)
# ============================================================
# Real PDFs don't have clean Markdown headers. We detect structure
# using heuristic patterns: ALL CAPS lines, numbered sections, etc.
# Falls back to recursive splitting for oversized sections.
# ============================================================

from langchain.text_splitter import RecursiveCharacterTextSplitter


def pdf_structure_chunker(documents, max_chunk_size=1000):
    """
    Detect section boundaries in PDF text using heuristic patterns,
    then split at those boundaries.
    
    Falls back to recursive splitting for sections that exceed max_chunk_size.
    
    Args:
        documents: list of LangChain Document objects (from PDF loader)
        max_chunk_size: maximum characters per chunk before triggering fallback
    
    Returns:
        list of Document objects with section metadata
    """
    all_chunks = []
    
    # Patterns that likely indicate section headers
    # 🎛️ CUSTOMIZE these for your specific documents!
    header_patterns = [
        r'^(?:CHAPTER|SECTION|ANNEX)\s+\d+',           # CHAPTER 1, SECTION 2
        r'^\d+\.\s+[A-Z]',                              # 1. Title format
        r'^[A-Z][A-Z\s]{10,}$',                         # ALL CAPS LINES (10+ chars)
        r'^(?:Strategic [Oo]bjective|Objective)\s+\d+',  # Strategic Objective 1
        r'^(?:Article|Appendix|Part)\s+\d+',             # Article 1, Appendix A
    ]
    
    # Fallback splitter for oversized sections
    fallback_splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chunk_size,
        chunk_overlap=50,
    )
    
    for doc in documents:
        lines = doc.page_content.split('\n')
        
        current_section = ""
        current_header = "Untitled Section"
        
        for line in lines:
            stripped = line.strip()
            
            # Check if this line looks like a header
            is_header = False
            for pattern in header_patterns:
                if re.match(pattern, stripped):
                    is_header = True
                    break
            
            if is_header and current_section.strip():
                # Save the previous section as a chunk
                chunk_doc = Document(
                    page_content=current_section.strip(),
                    metadata={
                        **doc.metadata,
                        "section_header": current_header,
                        "chunk_type": "document_aware_pdf",
                    }
                )
                
                # If section is too long, split it further with fallback
                if len(current_section) > max_chunk_size:
                    sub_chunks = fallback_splitter.split_documents([chunk_doc])
                    for sc in sub_chunks:
                        sc.metadata["section_header"] = current_header
                        sc.metadata["was_split"] = True  # flag that this was sub-split
                    all_chunks.extend(sub_chunks)
                else:
                    all_chunks.append(chunk_doc)
                
                # Start new section
                current_header = stripped
                current_section = ""
            else:
                current_section += line + "\n"
        
        # Don't forget the last section
        if current_section.strip():
            chunk_doc = Document(
                page_content=current_section.strip(),
                metadata={
                    **doc.metadata,
                    "section_header": current_header,
                    "chunk_type": "document_aware_pdf",
                }
            )
            if len(current_section) > max_chunk_size:
                sub_chunks = fallback_splitter.split_documents([chunk_doc])
                for sc in sub_chunks:
                    sc.metadata["section_header"] = current_header
                    sc.metadata["was_split"] = True
                all_chunks.extend(sub_chunks)
            else:
                all_chunks.append(chunk_doc)
    
    return all_chunks


# --- Run PDF structure chunker on our actual WHO document ---
# ------------------------------------------------------------------
# 🎛️ EXPERIMENT: Try changing max_chunk_size!
#    - 500: more granular sections
#    - 1000: balanced (good default)
#    - 2000: larger sections, fewer chunks
# ------------------------------------------------------------------
pdf_structure_chunks = pdf_structure_chunker(pages, max_chunk_size=1000)

# Evaluate both approaches
md_stats = evaluate_chunking_strategy(md_chunks, "Document-Aware (Markdown)")
pdf_stats = evaluate_chunking_strategy(pdf_structure_chunks, "Document-Aware (PDF Heuristic)")


### 🔍 What to Notice:
- **Markdown chunks** have clean header metadata (`h1`, `h2`, `h3`). This is the gold standard.
- **PDF heuristic chunks** are messier — the pattern matching isn't perfect. Some "headers" 
  might be false positives, and some real headers might be missed.
- Check the `section_header` metadata — this is incredibly useful for filtering at query time.
- Some chunks have `was_split: True` — these were sections too large for one chunk.

### 🧪 Try This:
1. Edit the `header_patterns` list to match YOUR documents' structure.
2. Add a pattern for your specific document format.
3. Try `max_chunk_size=500` — does splitting large sections improve retrieval?


---
## 🔴 Strategy 6: Code-Aware Chunking

**The idea:** Split code at function, class, or module boundaries — not at 
arbitrary line counts.

Code is NOT prose. Splitting a Python function in half produces meaningless 
embeddings. Code-aware chunking uses the **Abstract Syntax Tree (AST)** to 
identify meaningful boundaries.


In [ ]:
# ============================================================
# STRATEGY 6: Code-Aware Chunking
# ============================================================

import ast
from langchain.text_splitter import RecursiveCharacterTextSplitter

# --- Sample Python code to chunk ---
# In production, you'd load this from .py files in a repository
SAMPLE_CODE = '''
"""
AMR Data Analysis Module

This module provides utilities for analyzing antimicrobial resistance
surveillance data from WHO member states.
"""

import pandas as pd
import numpy as np
from datetime import datetime
from typing import List, Dict, Optional


class ResistanceTracker:
    """Tracks antimicrobial resistance rates across regions and time periods."""
    
    def __init__(self, data_source: str):
        """Initialize the tracker with a data source path."""
        self.data_source = data_source
        self.data = None
        self._cache = {}
    
    def load_data(self, start_year: int = 2015, end_year: int = 2024) -> pd.DataFrame:
        """
        Load resistance data for the specified year range.
        
        Args:
            start_year: First year to include
            end_year: Last year to include
            
        Returns:
            DataFrame with columns: country, pathogen, antibiotic, 
            resistance_rate, year, sample_size
        """
        self.data = pd.read_csv(self.data_source)
        self.data = self.data[
            (self.data['year'] >= start_year) & 
            (self.data['year'] <= end_year)
        ]
        return self.data
    
    def get_resistance_trend(self, pathogen: str, antibiotic: str, 
                              country: Optional[str] = None) -> Dict:
        """
        Calculate resistance trend for a specific pathogen-antibiotic combination.
        
        Returns dict with yearly rates and trend direction.
        """
        if self.data is None:
            raise ValueError("Data not loaded. Call load_data() first.")
        
        mask = (
            (self.data['pathogen'] == pathogen) & 
            (self.data['antibiotic'] == antibiotic)
        )
        if country:
            mask &= self.data['country'] == country
        
        subset = self.data[mask].groupby('year')['resistance_rate'].mean()
        
        trend = "increasing" if subset.iloc[-1] > subset.iloc[0] else "decreasing"
        
        return {
            'pathogen': pathogen,
            'antibiotic': antibiotic,
            'yearly_rates': subset.to_dict(),
            'trend': trend,
            'change': float(subset.iloc[-1] - subset.iloc[0]),
        }
    
    def identify_critical_combinations(self, threshold: float = 0.5) -> List[Dict]:
        """
        Identify pathogen-antibiotic combinations where resistance 
        exceeds the given threshold.
        """
        if self.data is None:
            raise ValueError("Data not loaded. Call load_data() first.")
        
        latest_year = self.data['year'].max()
        latest = self.data[self.data['year'] == latest_year]
        
        critical = latest[latest['resistance_rate'] > threshold]
        
        results = []
        for _, row in critical.iterrows():
            results.append({
                'pathogen': row['pathogen'],
                'antibiotic': row['antibiotic'],
                'country': row['country'],
                'resistance_rate': row['resistance_rate'],
                'year': latest_year,
            })
        
        return sorted(results, key=lambda x: x['resistance_rate'], reverse=True)


def calculate_regional_summary(data: pd.DataFrame, region: str) -> Dict:
    """
    Calculate summary statistics for a WHO region.
    
    Args:
        data: Full resistance dataset
        region: WHO region code (e.g., 'AFRO', 'EURO', 'SEARO')
    
    Returns:
        Dict with mean resistance rate, most resistant pathogen,
        and number of reporting countries.
    """
    regional = data[data['region'] == region]
    
    return {
        'region': region,
        'mean_resistance_rate': float(regional['resistance_rate'].mean()),
        'most_resistant_pathogen': regional.groupby('pathogen')['resistance_rate']
            .mean().idxmax(),
        'reporting_countries': regional['country'].nunique(),
        'total_samples': int(regional['sample_size'].sum()),
    }


def format_report(summary: Dict, trends: List[Dict]) -> str:
    """
    Format analysis results into a human-readable report.
    
    Args:
        summary: Output from calculate_regional_summary()
        trends: List of outputs from ResistanceTracker.get_resistance_trend()
    
    Returns:
        Formatted string report suitable for stakeholder communication.
    """
    report = f"AMR Report for {summary['region']}\\n"
    report += f"{'='*40}\\n"
    report += f"Mean resistance rate: {summary['mean_resistance_rate']:.1%}\\n"
    report += f"Most resistant pathogen: {summary['most_resistant_pathogen']}\\n"
    report += f"Reporting countries: {summary['reporting_countries']}\\n\\n"
    
    report += "Trends:\\n"
    for trend in trends:
        direction = "↑" if trend['trend'] == 'increasing' else "↓"
        report += f"  {direction} {trend['pathogen']} vs {trend['antibiotic']}: "
        report += f"{trend['change']:+.1%}\\n"
    
    return report
'''


# --- The Code-Aware Chunker ---

def code_aware_chunker(source_code, file_path="module.py", max_chunk_size=2000):
    """
    Parse Python source code using the AST (Abstract Syntax Tree)
    and split at function/class boundaries.
    
    Each function becomes a chunk. Each class becomes a chunk
    (or multiple chunks if it has many methods and exceeds max_chunk_size).
    Module-level code (imports, constants, docstrings) becomes its own chunk.
    
    Args:
        source_code: Python source code as a string
        file_path: Path to the file (for metadata)
        max_chunk_size: If a class is larger than this, split into methods
    
    Returns:
        list of Document objects (one per code unit)
    """
    chunks = []
    
    try:
        tree = ast.parse(source_code)
    except SyntaxError as e:
        # If we can't parse, fall back to recursive character splitting
        print(f"  ⚠️  AST parsing failed: {e}. Falling back to recursive splitting.")
        fallback = RecursiveCharacterTextSplitter(
            chunk_size=max_chunk_size, chunk_overlap=100
        )
        return fallback.create_documents(
            [source_code],
            metadatas=[{"source": file_path, "chunk_type": "code_fallback"}]
        )
    
    source_lines = source_code.split('\n')
    
    # Track which lines belong to top-level definitions
    defined_ranges = []
    
    for node in ast.iter_child_nodes(tree):
        
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            # --- Standalone function ---
            start_line = node.lineno - 1
            end_line = node.end_lineno
            func_code = '\n'.join(source_lines[start_line:end_line])
            
            docstring = ast.get_docstring(node) or ""
            
            chunks.append(Document(
                page_content=func_code,
                metadata={
                    "source": file_path,
                    "chunk_type": "function",
                    "name": node.name,
                    "docstring": docstring[:200],
                    "line_start": node.lineno,
                    "line_end": end_line,
                    "args": [arg.arg for arg in node.args.args],
                }
            ))
            defined_ranges.append((start_line, end_line))
        
        elif isinstance(node, ast.ClassDef):
            # --- Class definition ---
            start_line = node.lineno - 1
            end_line = node.end_lineno
            class_code = '\n'.join(source_lines[start_line:end_line])
            
            class_docstring = ast.get_docstring(node) or ""
            method_names = [
                n.name for n in ast.iter_child_nodes(node)
                if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
            ]
            
            # If the class is small enough, keep it as one chunk
            if len(class_code) <= max_chunk_size:
                chunks.append(Document(
                    page_content=class_code,
                    metadata={
                        "source": file_path,
                        "chunk_type": "class",
                        "name": node.name,
                        "docstring": class_docstring[:200],
                        "methods": method_names,
                        "line_start": node.lineno,
                        "line_end": end_line,
                    }
                ))
            else:
                # Class is too big — split into header + individual methods
                
                # 1. Capture the class header (definition + docstring + class-level code)
                first_method = None
                for child in ast.iter_child_nodes(node):
                    if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                        first_method = child
                        break
                
                if first_method:
                    header_end = first_method.lineno - 1
                    header_code = '\n'.join(source_lines[start_line:header_end])
                    if header_code.strip():
                        chunks.append(Document(
                            page_content=header_code,
                            metadata={
                                "source": file_path,
                                "chunk_type": "class_header",
                                "name": node.name,
                                "docstring": class_docstring[:200],
                                "methods": method_names,
                                "line_start": node.lineno,
                                "line_end": header_end,
                            }
                        ))
                
                # 2. Each method as its own chunk
                for child in ast.iter_child_nodes(node):
                    if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                        method_start = child.lineno - 1
                        method_end = child.end_lineno
                        method_code = '\n'.join(source_lines[method_start:method_end])
                        method_docstring = ast.get_docstring(child) or ""
                        
                        chunks.append(Document(
                            page_content=method_code,
                            metadata={
                                "source": file_path,
                                "chunk_type": "method",
                                "class_name": node.name,
                                "name": child.name,
                                "docstring": method_docstring[:200],
                                "line_start": child.lineno,
                                "line_end": method_end,
                                "args": [arg.arg for arg in child.args.args],
                            }
                        ))
            
            defined_ranges.append((start_line, end_line))
    
    # --- Capture module-level code (imports, constants, module docstring) ---
    module_lines = []
    for i, line in enumerate(source_lines):
        in_definition = any(start <= i < end for start, end in defined_ranges)
        if not in_definition:
            module_lines.append(line)
    
    module_code = '\n'.join(module_lines).strip()
    if module_code and len(module_code) > 20:
        chunks.append(Document(
            page_content=module_code,
            metadata={
                "source": file_path,
                "chunk_type": "module_level",
                "name": "module_header",
            }
        ))
    
    return chunks


# --- Run it ---
code_chunks = code_aware_chunker(SAMPLE_CODE, file_path="amr_analysis.py")

print(f"\n{'='*70}")
print(f"📊 STRATEGY: Code-Aware (AST-based)")
print(f"{'='*70}")
print(f"\n  Total chunks: {len(code_chunks)}")

for i, chunk in enumerate(code_chunks):
    chunk_type = chunk.metadata.get('chunk_type', 'unknown')
    name = chunk.metadata.get('name', 'unnamed')
    class_name = chunk.metadata.get('class_name', '')
    full_name = f"{class_name}.{name}" if class_name else name
    
    print(f"\n  📄 Chunk {i}: [{chunk_type}] {full_name}")
    print(f"     Size: {len(chunk.page_content)} chars | "
          f"Lines: {chunk.metadata.get('line_start', '?')}-{chunk.metadata.get('line_end', '?')}")
    
    # Show metadata highlights
    if 'args' in chunk.metadata:
        print(f"     Args: {chunk.metadata['args']}")
    if 'methods' in chunk.metadata:
        print(f"     Methods: {chunk.metadata['methods']}")
    
    # Show first 4 lines of code
    code_lines = chunk.page_content.split('\n')
    for line in code_lines[:4]:
        print(f"     {line}")
    total_lines = len(chunk.page_content.split('\n'))
    if total_lines > 4:
        print(f"     ... ({total_lines} lines total)")

# --- Evaluate with code-specific queries ---
CODE_QUERIES = [
    "How do I calculate resistance trends for a specific pathogen?",
    "What function formats the analysis into a report?",
    "How does the ResistanceTracker class load data?",
    "How do I find critical pathogen-antibiotic combinations?",
    "What does the regional summary function return?",
]

code_stats = evaluate_chunking_strategy(
    code_chunks, "Code-Aware (AST)", queries=CODE_QUERIES
)


### 🔍 What to Notice:
- Each chunk is a **complete, meaningful unit**: a function, a method, or a class.
- The metadata is incredibly rich: function names, arguments, docstrings, line numbers.
- No chunk starts mid-function or ends mid-class.
- The module-level chunk captures imports and the module docstring separately.

### 🧪 Try This:
1. Change `max_chunk_size` to 500 — the `ResistanceTracker` class will get split into 
   individual methods instead of staying as one chunk.
2. Paste your own Python code into `SAMPLE_CODE` and see how it gets chunked.
3. Try adding a very large function (100+ lines) — does the chunker handle it well?


---
---
## 🏆 Grand Comparison: All Strategies Head-to-Head

Now let's put all six strategies side by side. Same document. Same embedding model. 
Same queries. Different chunking. 

**This is the cell that answers: "Which strategy works best for MY document?"**


In [ ]:
# ============================================================
# 🏆 GRAND COMPARISON: All Strategies Side by Side
# ============================================================

print("\n" + "🏆" * 35)
print("  GRAND COMPARISON: All Chunking Strategies")
print("🏆" * 35)

# Collect all strategies (excluding code — different document type)
strategies = {
    "1. Fixed-Size (500)":        fixed_chunks,
    "2. Recursive Char (500)":    recursive_chunks,
    "3. Sentence-Window (w=3)":   sentence_window_chunks,
    "4. Semantic (t=0.75)":       semantic_chunks,
    "5. Doc-Aware (PDF)":         pdf_structure_chunks,
}

# --- Summary Table ---
print(f"\n{'Strategy':<30} {'Chunks':>8} {'Avg Size':>10} {'Min':>8} {'Max':>8} {'Std Dev':>10}")
print("─" * 80)

for name, chunks in strategies.items():
    sizes = [len(c.page_content) for c in chunks]
    avg = sum(sizes) // len(sizes)
    std = int(np.std(sizes))
    print(f"{name:<30} {len(chunks):>8} {avg:>10} {min(sizes):>8} {max(sizes):>8} {std:>10}")

# --- Head-to-Head Retrieval Comparison ---
COMPARISON_QUERIES = [
    "What are the strategic objectives of the global action plan?",
    "How does antimicrobial resistance spread between animals and humans?",
    "What does the WHO recommend for surveillance of resistant infections?",
]

print(f"\n\n{'='*80}")
print("📊 HEAD-TO-HEAD RETRIEVAL COMPARISON")
print(f"{'='*80}")

for query in COMPARISON_QUERIES:
    print(f"\n🔍 Query: \"{query}\"")
    print(f"{'─'*80}")

    for name, chunks in strategies.items():
        # In-memory store — no disk writes, no SQLite errors
        chroma_client = chromadb.EphemeralClient()
        vs = Chroma.from_documents(
            documents=chunks,
            embedding=embedding_model,
            client=chroma_client,
        )

        results = vs.similarity_search_with_score(query, k=1)
        doc, score = results[0]
        preview = doc.page_content[:100].replace('\n', ' ')

        # Visual score bar (L2 distance — lower is better)
        bar_length = max(1, int((2 - score) * 20))
        bar = "█" * bar_length

        print(f"  {name:<30} dist={score:.4f} {bar}")
        print(f"  {'':>30} \"{preview}...\"")
        # EphemeralClient cleaned up automatically


### 🔍 What to Notice in the Grand Comparison:
1. **Chunk count varies wildly** — sentence-window produces the most chunks (one per sentence), 
   while document-aware might produce the fewest.
2. **Std Dev reveals consistency** — fixed-size has low std dev (uniform chunks), 
   while semantic and document-aware have high std dev (variable chunks).
3. **Same query, different top results** — each strategy surfaces different content. 
   Which one is most relevant to the actual question?
4. **Distance scores differ** — lower is better in Chroma. But a lower score doesn't 
   always mean a better *answer*. The content matters more than the score.


---
## 🎯 The Decision Flowchart: Which Strategy Should YOU Use?
### The Key Principle:
> **Start simple. Measure. Upgrade only when your metrics justify the complexity.**


---
## 🧪 Exercises: Your Turn!

Now that you've seen all six strategies, try these experiments:

### Exercise 1: Chunk Size Sensitivity
Go back to Strategy 2 (Recursive Character) and try these chunk sizes: 
200, 500, 800, 1500. Which gives the best retrieval results for the test queries?

### Exercise 2: Your Own Document  
Replace `who_amr_action_plan.pdf` with a document from YOUR project. 
Run all strategies. Which one works best for your content?

### Exercise 3: Custom Queries
Replace `TEST_QUERIES` with questions that matter for YOUR use case. 
Do the strategy rankings change?

### Exercise 4: Semantic Threshold Tuning
Run the semantic chunker with thresholds: 0.5, 0.65, 0.75, 0.85, 0.95.
Use the similarity landscape visualization to understand why.

### Exercise 5: Hybrid Strategy
Build a chunker that uses document-aware splitting for structured sections 
and recursive character splitting for unstructured sections. 
Hint: check the `pdf_structure_chunker` — it already does this with its fallback!


In [ ]:
# ============================================================
# 🧹 CLEANUP: Remove any leftover Chroma databases
# ============================================================

import glob

for db_dir in glob.glob("./chroma_*"):
    if os.path.isdir(db_dir):
        shutil.rmtree(db_dir, ignore_errors=True)

print("✅ Cleanup complete! All temporary vector stores removed.")
print("\n🎉 You've completed the Chunking Strategies notebook!")
print("   Go forth and chunk wisely. 🔪")
